# WHACS offshore waves at the goals

Reads WHACS (bulk and the four partitions: 0 = wind sea, 1-3 = swells) at the
WHACS node of every goal from the local copy of the global archive, cleans the
peak periods and writes the processed zarr used by the dynamic runs and the
reconstruction. The partitions are written as WHACS gives them: the energy they
miss is measured, and closed on the bulk, by the calibration (07b).

The archive is the `archive` of the WHACS `offshore_waves` entry: one folder per
variable with monthly files on the global SMC mesh, as CSIRO publishes them
([doi:10.25919/shdk-7p29](https://doi.org/10.25919/shdk-7p29)). Only the file
chunks that hold the goal nodes are read (`utils.whacs`).

**The archive is not part of HyGloW.** On the GeoOcean cluster it is
`/lustre/geocean/DATA/CSIRO/WHACS/global-smc`. Elsewhere, download the 20
variables read here (`hs`, `fp`, `dir`, `spr` and `phs`, `ptp`, `pdir`, `pspr`
0-3; about 14 TB for 1979-2026) or ask GeoOcean for the goal series of your case.

**Prerequisites:** `04_Select_Goals_by_Hindcasts.ipynb` (`<Case>_Goals.json`,
with a `whacs` node per goal).
**Next:** `07b_Satellite_Calibration.ipynb` validates (and calibrates) this
hindcast against satellite altimetry.

## Configuration


In [1]:
import xarray as xr

from utils import whacs
from utils.config import get_offshore_wave, resolve

offshore_name = "WHACS"  # offshore_waves entry with the archive
period = None  # months to read, e.g. slice("1979", "2023"); None = the whole archive
workers = 16  # processes reading the monthly files

archive = whacs.archive_path(offshore_name)
print(f"archive: {archive}")
print(f"output : {resolve(get_offshore_wave(offshore_name)['data_path'])}")

archive: /lustre/geocean/DATA/CSIRO/WHACS/global-smc
output : /lustre/geocean/WORK/users/javiert/projects/HyGloW/NewSouthWales/inputs/WHACS/processed/NewSouthWales_WHACS_goal.zarr


## Goal nodes

The archive seapoint of each goal's WHACS node (goals JSON `hindcast.whacs`);
`distance` (degrees) is ~0 when the node is a WHACS seapoint.

In [2]:
whacs.goal_seapoints(archive, offshore_name)

,seapoint,longitude,latitude,distance
goal,,,,
1,923284,153.56250,-28.06250,0.0
2,937305,154.03125,-27.84375,0.0
3,937042,154.15625,-28.21875,0.0
4,936603,154.28125,-28.84375,0.0
5,936336,154.28125,-29.21875,0.0
6,935981,154.15625,-29.71875,0.0
7,935621,154.15625,-30.21875,0.0
8,935343,153.90625,-30.59375,0.0
9,934970,153.65625,-31.09375,0.0


## Read the archive

Every variable of every month in `period` at the goal seapoints (`tp = 1 / fp`).

In [3]:
whacs_data = whacs.whacs_at_goals(offshore_name, period=period, workers=workers)
whacs_data

<xarray.Dataset> Size: 2GB
Dimensions:   (time: 412752, goal: 26)
Coordinates:
  * time      (time) datetime64[ns] 3MB 1979-01-01 ... 2026-01-31T23:00:00
  * goal      (goal) int64 208B 1 2 3 4 5 6 7 8 9 ... 18 19 20 21 22 23 24 25 26
    seapoint  (goal) int64 208B 923284 937305 937042 ... 930128 929894 922215
Data variables: (12/21)
    hs        (time, goal) float64 86MB 1.366 1.73 1.658 ... 2.148 2.15 2.07
    fp        (time, goal) float64 86MB 0.112 0.117 0.116 ... 0.172 0.171 0.174
    dir       (time, goal) float64 86MB 104.3 124.3 121.7 ... 189.9 203.2 199.0
    spr       (time, goal) float64 86MB 32.0 40.4 41.1 42.5 ... 55.4 54.2 50.9
    phs0      (time, goal) float64 86MB 0.756 1.336 1.216 ... 1.9 1.928 1.89
    ptp0      (time, goal) float64 86MB 6.23 6.71 6.75 6.69 ... 5.7 5.75 5.63
    ...        ...
    pspr2     (time, goal) float64 86MB 9.2 9.3 8.7 10.7 ... 11.0 10.6 10.7 11.8
    phs3      (time, goal) float64 86MB nan 0.126 0.118 ... 0.412 0.302 nan
    ptp3      (time, goal) float64 86MB nan 10.39 12.57 ... 11.44 11.68 nan
    pdir3     (time, goal) float64 86MB nan 34.7 38.6 3.0 ... nan 59.5 59.2 nan
    pspr3     (time, goal) float64 86MB nan 9.2 11.4 7.9 ... nan 11.1 12.3 nan
    tp        (time, goal) float64 86MB 8.929 8.547 8.621 ... 5.814 5.848 5.747
Attributes: (12/54)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lat_max:              74.90625
    geospatial_lon_min:              -180.0
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_resolution:        P0DT1H0M0S
    hywaves_source:                  WHACS global archive /lustre/geocean/DAT...

Bulk `tp = 1 / fp` explodes when the spectral peak falls below the WHACS
frequency grid (hours with Tp of 50-250 s). Those take the peak period of the
most energetic partition (`clean_peak_periods`); partition periods above
`max_tp` become NaN.

In [4]:
from bluemath_tk.waves.hywaves.offshore import clean_peak_periods

max_tp = 25.0  # s, largest valid peak period
whacs_data = clean_peak_periods(whacs_data, max_tp=max_tp)
print(f"bulk tp replaced: {whacs_data.attrs['tp_replaced']}, partition tp removed: {whacs_data.attrs['ptp_removed']}")

bulk tp replaced: 10, partition tp removed: 0


## Write the zarr

In [5]:
whacs_data.to_zarr(resolve(get_offshore_wave(offshore_name)["data_path"]), mode="w")

In [6]:
xr.open_zarr(resolve(get_offshore_wave(offshore_name)["data_path"])).load()

<xarray.Dataset> Size: 2GB
Dimensions:   (time: 412752, goal: 26)
Coordinates:
  * time      (time) datetime64[ns] 3MB 1979-01-01 ... 2026-01-31T23:00:00
  * goal      (goal) int64 208B 1 2 3 4 5 6 7 8 9 ... 18 19 20 21 22 23 24 25 26
    seapoint  (goal) int64 208B 923284 937305 937042 ... 930128 929894 922215
Data variables: (12/21)
    dir       (time, goal) float64 86MB 104.3 124.3 121.7 ... 189.9 203.2 199.0
    fp        (time, goal) float64 86MB 0.112 0.117 0.116 ... 0.172 0.171 0.174
    hs        (time, goal) float64 86MB 1.366 1.73 1.658 ... 2.148 2.15 2.07
    pdir0     (time, goal) float64 86MB 144.2 151.2 154.5 ... 200.4 209.3 205.1
    pdir1     (time, goal) float64 86MB 88.3 86.3 86.0 84.7 ... 32.3 31.8 48.0
    pdir2     (time, goal) float64 86MB 37.4 36.9 35.1 34.7 ... 61.8 60.7 60.1
    ...        ...
    ptp0      (time, goal) float64 86MB 6.23 6.71 6.75 6.69 ... 5.7 5.75 5.63
    ptp1      (time, goal) float64 86MB 9.04 9.11 9.12 9.09 ... 6.65 6.64 7.22
    ptp2      (time, goal) float64 86MB 10.43 12.37 10.89 ... 7.36 7.39 11.66
    ptp3      (time, goal) float64 86MB nan 10.39 12.57 ... 11.44 11.68 nan
    spr       (time, goal) float64 86MB 32.0 40.4 41.1 42.5 ... 55.4 54.2 50.9
    tp        (time, goal) float64 86MB 8.929 8.547 8.621 ... 5.814 5.848 5.747
Attributes: (12/56)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_resolution:        P0DT1H0M0S
    hywaves_source:                  WHACS global archive /lustre/geocean/DAT...
    tp_replaced:                     10
    ptp_removed:                     0